## Notebook 10 — Water-balance implications of AET method choice

**Why this matters:** in the NPS WBM's bucket structure, actual evapotranspiration
(AET) is the model's primary *outflow* competing with runoff generation. Concretely
(see `wbm/snow_soil.py`): each day, `storage_add = W - AET - dSOIL` (where `W` is rain
+ snowmelt reaching the soil), and `RUNOFF = storage_release(storage_add) + direct_frac
* RAIN`. Higher AET means less water left over for storage release, so **at the
annual-to-multiyear scale, underestimating AET directly inflates the WBM's runoff
estimate** (soil-moisture and snowpack carry-over roughly net to zero over several
years, so `RUNOFF ≈ PPT - AET` is a good long-run approximation, and this notebook
checks that against the model's own literal `RUNOFF` output rather than assuming it).

Notebooks 04/06/08 established that **default (uncalibrated) Oudin systematically
underestimates AET** relative to OpenET and flux towers, and that **per-site-calibrated
Oudin** and **Penman-Monteith** both correct much of that bias. This notebook asks the
practical follow-on question: **if the WBM is currently being run with uncalibrated
Oudin, how much is its runoff estimate being inflated as a result — and how much of
that gets corrected by switching to per-site-calibrated Oudin vs. switching to
Penman-Monteith?**

This is a read-only analysis: it reuses the full 2016–2023 daily WBM outputs already
cached by notebooks 03 (default Oudin), 04 (per-site-calibrated Oudin), and 06
(Penman-Monteith) — no new WBM runs, no GEE access. `RUNOFF` is already a per-day output
column in each of those caches, so this notebook just aggregates and compares it
directly, rather than re-deriving it from `PPT - AET`.

**Prerequisites:** run notebooks 01→02→03 at least once (default Oudin WBM cache),
notebook 04 at least once with its per-site calibration cell (produces
`Data/gridmet_cache/wbm_results_site_cal_{OBJECTIVE}.csv`), and notebook 06 at least once
(`Data/gridmet_cache/wbm_results_penman_monteith.csv`).

# Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 130)
os.makedirs("../Data/runoff_implications", exist_ok=True)
print("Imports OK -- this notebook only reads cached CSVs, no GEE/WBM run needed.")

### Load cached full-period WBM outputs (default Oudin, per-site-calibrated Oudin, Penman-Monteith)

All three caches store fluxes in **inches** (the repo's default `TO_INCHES = True`),
despite the `_mm`-suffixed column names -- converted to mm here (`x 25.4`) to match the
units used everywhere else in this project's reports.

In [ ]:
IN_TO_MM = 25.4
COLS = ["date", "site", "ecosystem", "state", "ppt_mm", "AET", "RUNOFF"]

with open("../Data/gridmet_cache/last_objective.txt") as f:
    CAL_OBJECTIVE = f.read().strip()

wbm_default = pd.read_csv(
    "../Data/gridmet_cache/wbm_results_flux_towers_2016_2023.csv",
    usecols=COLS, parse_dates=["date"],
)
wbm_site_cal = pd.read_csv(
    f"../Data/gridmet_cache/wbm_results_site_cal_{CAL_OBJECTIVE}.csv",
    usecols=COLS, parse_dates=["date"],
)
wbm_pm = pd.read_csv(
    "../Data/gridmet_cache/wbm_results_penman_monteith.csv",
    usecols=COLS, parse_dates=["date"],
)

for _df in (wbm_default, wbm_site_cal, wbm_pm):
    for _c in ("ppt_mm", "AET", "RUNOFF"):
        _df[_c] = _df[_c] * IN_TO_MM

print(f"wbm_default  (uncalibrated Oudin)      : {wbm_default.shape}")
print(f"wbm_site_cal (per-site cal. Oudin, {CAL_OBJECTIVE}) : {wbm_site_cal.shape}")
print(f"wbm_pm       (Penman-Monteith)          : {wbm_pm.shape}")

### Aggregate to mean annual water balance per site

Sum daily `ppt_mm` / `AET` / `RUNOFF` to calendar-year totals per site, then average
across all available years -- this is the "typical annual water balance" each variant
implies at each site.

In [ ]:
def mean_annual_balance(df, label):
    """Calendar-year sums per site, then averaged across years -> one row per site."""
    yearly = (
        df.assign(year=df["date"].dt.year)
        .groupby(["site", "year"], as_index=False)[["ppt_mm", "AET", "RUNOFF"]]
        .sum()
    )
    out = (
        yearly.groupby("site", as_index=False)[["ppt_mm", "AET", "RUNOFF"]]
        .mean()
        .rename(columns={
            "ppt_mm": f"ppt_{label}", "AET": f"aet_{label}", "RUNOFF": f"runoff_{label}",
        })
    )
    return out


ann_default  = mean_annual_balance(wbm_default,  "default")
ann_site_cal = mean_annual_balance(wbm_site_cal, "sitecal")
ann_pm       = mean_annual_balance(wbm_pm,       "pm")

eco_lookup = wbm_default[["site", "ecosystem", "state"]].drop_duplicates()

balance = (
    ann_default
    .merge(ann_site_cal, on="site", how="inner")
    .merge(ann_pm,       on="site", how="inner")
    .merge(eco_lookup,   on="site", how="left")
)

# Sanity check: RUNOFF ~ PPT - AET at the multi-year mean scale (storage/snowpack
# carry-over should mostly net out over 8 years) -- this is NOT assumed anywhere
# above, just checked here against the model's own literal RUNOFF output.
for label in ("default", "sitecal", "pm"):
    resid = (balance[f"ppt_{label}"] - balance[f"aet_{label}"]) - balance[f"runoff_{label}"]
    print(f"{label:8s}: PPT - AET vs RUNOFF, mean residual = {resid.mean():+.1f} mm/yr "
          f"(median abs residual = {resid.abs().median():.1f} mm/yr, n={len(balance)} sites)")

print(f"\nSites with all three variants matched: {len(balance)}")
balance.to_csv("../Data/runoff_implications/annual_water_balance_by_site.csv", index=False)

### How much is uncalibrated Oudin currently overestimating runoff?

`delta_runoff = runoff_default - runoff_{alternative}` is how much runoff would
**decrease** (i.e. how much the current uncalibrated-Oudin estimate is being
**overestimated**) if the WBM switched to that alternative. Reported both in mm/year and
as a percentage of the alternative's own runoff estimate (a natural "how far off is the
status quo" framing) and as a percentage of mean annual precipitation (a scale-free
comparison across sites with very different climates).

In [ ]:
balance["delta_runoff_sitecal"] = balance["runoff_default"] - balance["runoff_sitecal"]
balance["delta_runoff_pm"]      = balance["runoff_default"] - balance["runoff_pm"]

# "% of the alternative's own runoff estimate" is a natural "how far off is the status
# quo" framing, but blows up at sites where the alternative's runoff is itself near
# zero (e.g. very dry sites) -- restrict that specific ratio to sites with a
# meaningfully nonzero denominator (>= 10 mm/yr) so a handful of near-zero-runoff sites
# don't dominate the mean. The %-of-precipitation version below has no such issue
# (precipitation is never near zero) and is reported for all sites.
RUNOFF_FLOOR = 10.0  # mm/yr

balance["pct_overest_sitecal_of_runoff"] = np.where(
    balance["runoff_sitecal"] >= RUNOFF_FLOOR,
    100 * balance["delta_runoff_sitecal"] / balance["runoff_sitecal"],
    np.nan,
)
balance["pct_overest_pm_of_runoff"] = np.where(
    balance["runoff_pm"] >= RUNOFF_FLOOR,
    100 * balance["delta_runoff_pm"] / balance["runoff_pm"],
    np.nan,
)
balance["pct_overest_sitecal_of_ppt"] = (
    100 * balance["delta_runoff_sitecal"] / balance["ppt_default"].replace(0, np.nan)
)
balance["pct_overest_pm_of_ppt"] = (
    100 * balance["delta_runoff_pm"] / balance["ppt_default"].replace(0, np.nan)
)

print("── Nationwide: how much does uncalibrated-Oudin runoff shrink if switched? ──")
for label, alt in [("per-site-calibrated Oudin", "sitecal"), ("Penman-Monteith", "pm")]:
    d_mm   = balance[f"delta_runoff_{alt}"]
    d_pctR = balance[f"pct_overest_{alt}_of_runoff"].dropna()
    d_pctP = balance[f"pct_overest_{alt}_of_ppt"]
    n_excluded = balance[f"runoff_{alt}"].lt(RUNOFF_FLOOR).sum()
    print(f"\nSwitch default Oudin -> {label}:")
    print(f"  mean  Δrunoff = {d_mm.mean():+7.1f} mm/yr   "
          f"median Δrunoff = {d_mm.median():+7.1f} mm/yr")
    print(f"  mean  overestimate = {d_pctR.mean():+6.1f}% of the {label} runoff estimate  "
          f"(median = {d_pctR.median():+6.1f}%, {n_excluded} near-zero-runoff site(s) excluded)")
    print(f"  mean  overestimate = {d_pctP.mean():+6.1f}% of mean annual precip  "
          f"(median = {d_pctP.median():+6.1f}%)")

balance.to_csv("../Data/runoff_implications/annual_water_balance_by_site.csv", index=False)
print("\nSaved to Data/runoff_implications/annual_water_balance_by_site.csv")

### By ecosystem

In [ ]:
eco_cols = ["ppt_default", "aet_default", "aet_sitecal", "aet_pm",
            "runoff_default", "runoff_sitecal", "runoff_pm",
            "delta_runoff_sitecal", "delta_runoff_pm",
            "pct_overest_sitecal_of_ppt", "pct_overest_pm_of_ppt"]
eco_summary = (
    balance.groupby("ecosystem")[eco_cols].mean().round(1)
    .join(balance.groupby("ecosystem").size().rename("n_sites"))
    .reset_index()
    .sort_values("n_sites", ascending=False)
)
print(eco_summary.to_string(index=False))
eco_summary.to_csv("../Data/runoff_implications/annual_water_balance_by_ecosystem.csv", index=False)

x = np.arange(len(eco_summary))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4.2), constrained_layout=True)
ax.bar(x - w/2, eco_summary["pct_overest_sitecal_of_ppt"], width=w,
       color="seagreen", label="Switch to per-site-calibrated Oudin")
ax.bar(x + w/2, eco_summary["pct_overest_pm_of_ppt"], width=w,
       color="tomato", label="Switch to Penman-Monteith")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(eco_summary["ecosystem"], rotation=30, ha="right", fontsize=8.5)
ax.set_ylabel("Mean runoff overestimate from using\nuncalibrated Oudin (% of annual precip)", fontsize=9)
ax.set_title("How much does uncalibrated Oudin overestimate annual runoff?\nby ecosystem, as % of mean annual precipitation",
             fontsize=10.5, fontweight="bold")
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig("../Data/runoff_implications/runoff_overestimate_by_ecosystem.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to Data/runoff_implications/runoff_overestimate_by_ecosystem.png")

### Site-level comparison: annual runoff, default Oudin vs. the two alternatives

In [ ]:
ecosystems_all = sorted(balance["ecosystem"].dropna().unique())
tab_colors = plt.cm.tab10.colors
eco_colors = {e: tab_colors[i % 10] for i, e in enumerate(ecosystems_all)}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), constrained_layout=True)
panels = [
    (axes[0], "runoff_sitecal", "Per-site-calibrated Oudin"),
    (axes[1], "runoff_pm",      "Penman-Monteith"),
]
for ax, col, label in panels:
    for eco, grp in balance.groupby("ecosystem"):
        ax.scatter(grp[col], grp["runoff_default"], s=45, color=eco_colors.get(eco, "grey"),
                   edgecolors="k", linewidths=0.4, alpha=0.85, label=eco, zorder=3)
    lims = [0, max(balance[col].max(), balance["runoff_default"].max()) * 1.08]
    ax.plot(lims, lims, color="grey", linestyle="--", linewidth=1, zorder=1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel(f"{label} annual runoff (mm/yr)", fontsize=9)
    ax.set_ylabel("Uncalibrated-Oudin annual runoff (mm/yr)", fontsize=9)
    ax.set_title(f"Uncalibrated Oudin vs. {label}", fontsize=10, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

handles = [plt.Line2D([], [], marker="o", linestyle="", markersize=7,
                      markerfacecolor=eco_colors[e], markeredgecolor="k", label=e)
           for e in ecosystems_all]
fig.legend(handles=handles, loc="lower center", ncol=min(len(ecosystems_all), 4),
           bbox_to_anchor=(0.5, -0.08), fontsize=8, frameon=False)
fig.suptitle("Points above the 1:1 line = uncalibrated Oudin overestimates annual runoff at that site",
             fontsize=10.5, fontweight="bold")
plt.savefig("../Data/runoff_implications/runoff_scatter_default_vs_alternatives.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved to Data/runoff_implications/runoff_scatter_default_vs_alternatives.png")

n_above_sitecal = (balance["runoff_default"] > balance["runoff_sitecal"]).sum()
n_above_pm      = (balance["runoff_default"] > balance["runoff_pm"]).sum()
print(f"\nSites where uncalibrated Oudin overestimates runoff vs. per-site-cal Oudin: "
      f"{n_above_sitecal} / {len(balance)}")
print(f"Sites where uncalibrated Oudin overestimates runoff vs. Penman-Monteith:      "
      f"{n_above_pm} / {len(balance)}")

### Takeaways

- At the annual scale, the model's own `RUNOFF` output tracks `PPT - AET` reasonably
  well but not exactly -- a consistent small positive residual (mean +38, +16, and
  +30 mm/yr for default, per-site-calibrated, and Penman-Monteith respectively, out of
  300-1000 mm/yr typical annual precipitation) shows some water is retained in
  storage/snowpack carry-over that hasn't fully released within this 8-year window,
  rather than a numerical error -- the headline numbers below use the model's literal
  `RUNOFF` output directly, not the `PPT - AET` approximation.
- **Because uncalibrated Oudin underestimates AET nationwide, it correspondingly
  overestimates annual runoff** at most sites: switching to per-site-calibrated Oudin
  reduces the runoff estimate at 45 of 72 sites (mean correction +113.9 mm/yr, median
  +39.2 mm/yr, or +13.4%/+4.8% of mean annual precipitation); switching to
  Penman-Monteith reduces it at 60 of 72 sites (mean correction +207.6 mm/yr, median
  +128.4 mm/yr, or +25.5%/+20.7% of mean annual precipitation).
- **Penman-Monteith implies a substantially larger downward runoff correction than
  calibrated Oudin, in every ecosystem** -- not because it's more accurate (notebook 08
  found calibrated Oudin has better RMSE/R² against both OpenET and flux towers), but
  because Penman-Monteith's AET runs consistently less negatively biased (i.e. higher)
  than calibrated Oudin's against both references. A method can have less bias (closer
  average AET, and so a bigger apparent "correction" to today's overestimated runoff)
  while still being less accurate overall (worse RMSE/R², meaning it gets the timing
  and site-to-site pattern more wrong) -- so "biggest runoff correction" and "most
  trustworthy AET method" are not the same question, and this notebook only answers the
  first one.
- The size of the correction varies a lot by ecosystem: largest in Mixed Forests and
  Evergreen Forests (where the underlying AET bias was also largest), smallest -- and
  even reversed for per-site-calibrated Oudin -- in Shrublands and Wetland/Riparian,
  where per-site calibration barely changes AET at all (consistent with notebook 08's
  finding that calibration doesn't help much in those ecosystems either).
- This notebook reports the *typical annual* water-balance effect. A site with high
  year-to-year variability could see a different seasonal pattern of over/under-runoff
  than its multi-year average suggests -- worth a follow-up if a specific park's monthly
  or peak-flow runoff estimate is the actual quantity of interest, not just the annual
  total.